<a href="https://colab.research.google.com/github/Derio13/Group3A-School-Results-/blob/main/notebooks/Group3A_School_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Group 3A School Results — Data Cleaning

 Week 3: Python Data Cleaning

Project: School Results Analysis  
Group: Group 3A  
Source Schema: `raw_school`  
Cleaned Schema: `group3a` Objective
Clean and prepare the school results dataset for modelling and analysis in Power BI.

### Main Data Quality Issues Identified
- Mixed date formats
- Inconsistent term labels
- Missing score values
- Invalid scores outside the 0–100 range
- Duplicate result records
- Orphan student references
- Inconsistent sex and region labels
- Missing student region values

### Tools Used
- PostgreSQL
- DBeaver
- Google Colab
- Python
- pandas
- psycopg2
- SQLAlchemy

In [5]:
!pip install psycopg2-binary

In [7]:
import psycopg2
from getpass import getpass

host = "internship-db.coh86gwewtxb.us-east-1.rds.amazonaws.com"
port = "5432"
database = "internship"
username = "group3a"

password = getpass("Enter database password: ")

conn = psycopg2.connect(
    host=host,
    port=port,
    database=database,
    user=username,
    password=password,
    sslmode="require"
)

print("Database connection successful!")

Enter database password: ··········
Database connection successful!


In [8]:
import pandas as pd

results = pd.read_sql("SELECT * FROM raw_school.results", conn)
students = pd.read_sql("SELECT * FROM raw_school.students", conn)
subjects = pd.read_sql("SELECT * FROM raw_school.subjects", conn)
teachers = pd.read_sql("SELECT * FROM raw_school.teachers", conn)

print("results:", results.shape)
print("students:", students.shape)
print("subjects:", subjects.shape)
print("teachers:", teachers.shape)

/tmp/ipykernel_1484/3684876294.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  results = pd.read_sql("SELECT * FROM raw_school.results", conn)
/tmp/ipykernel_1484/3684876294.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  students = pd.read_sql("SELECT * FROM raw_school.students", conn)
/tmp/ipykernel_1484/3684876294.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  subjects = pd.read_sql("SELECT * FROM raw_school.subjects", conn)


results: (30120, 8)
students: (1200, 6)
subjects: (10, 3)
teachers: (50, 4)


/tmp/ipykernel_1484/3684876294.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  teachers = pd.read_sql("SELECT * FROM raw_school.teachers", conn)


In [9]:
results_clean = results.copy()

results_clean.head()

,result_id,exam_date,student_id,subject_id,teacher_id,term,score,attendance_pct
0,5286,2024-01-05,633,9,44,Term 1,NaN,75.1
1,11177,18/05/2025,421,9,50,Term 2,57.1,94.4
2,10157,2024-05-04,222,3,22,Term 3,48.9,83.2
3,16448,2025-06-21,961,10,42,Term 3,41.5,94.4
4,29013,2025-09-29,190,9,23,Term 3,27.2,94.4


In [10]:
results_clean["term"] = (
    results_clean["term"]
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

results_clean["term"].value_counts()

,count
term,
Term 2,10168
Term 3,10017
Term 1,9935


In [11]:
results_clean["exam_date_clean"] = pd.to_datetime(
    results_clean["exam_date"],
    errors="coerce",
    dayfirst=True
)

print(results_clean[["exam_date", "exam_date_clean"]].head(10))

print(
    "Unparsed dates:",
    results_clean["exam_date_clean"].isna().sum()
)

    exam_date exam_date_clean
0  2024-01-05      2024-05-01
1  18/05/2025             NaT
2  2024-05-04      2024-04-05
3  2025-06-21             NaT
4  2025-09-29             NaT
5  2025-07-19             NaT
6  2024-10-03      2024-03-10
7  2024-03-09      2024-09-03
8  2024-12-21             NaT
9  2025-05-04      2025-04-05
Unparsed dates: 20415


In [12]:
from datetime import datetime
import pandas as pd

def parse_mixed_date(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    formats = [
        "%Y-%m-%d",   # 2024-01-05
        "%Y/%m/%d",   # 2024/01/05
        "%d-%b-%Y",   # 05-Jan-2024
        "%d/%m/%Y"    # 05/01/2024
    ]

    for fmt in formats:
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            continue

    return pd.NaT

results_clean["exam_date_clean"] = results_clean["exam_date"].apply(parse_mixed_date)

print(results_clean[["exam_date", "exam_date_clean"]].head(10))
print("Unparsed dates:", results_clean["exam_date_clean"].isna().sum())

    exam_date exam_date_clean
0  2024-01-05      2024-01-05
1  18/05/2025      2025-05-18
2  2024-05-04      2024-05-04
3  2025-06-21      2025-06-21
4  2025-09-29      2025-09-29
5  2025-07-19      2025-07-19
6  2024-10-03      2024-10-03
7  2024-03-09      2024-03-09
8  2024-12-21      2024-12-21
9  2025-05-04      2025-05-04
Unparsed dates: 0


In [13]:
results_clean.loc[
    (results_clean["score"] < 0) |
    (results_clean["score"] > 100),
    "score"
] = pd.NA

print("Missing scores after cleaning:", results_clean["score"].isna().sum())

Missing scores after cleaning: 1562


In [14]:
before_rows = len(results_clean)

results_clean = results_clean.drop_duplicates(
    subset=[
        "exam_date_clean",
        "student_id",
        "subject_id",
        "teacher_id",
        "term",
        "score",
        "attendance_pct"
    ],
    keep="first"
)

after_rows = len(results_clean)

print("Rows before duplicate removal:", before_rows)
print("Rows after duplicate removal:", after_rows)
print("Duplicates removed:", before_rows - after_rows)

Rows before duplicate removal: 30120
Rows after duplicate removal: 30000
Duplicates removed: 120


In [15]:
valid_student_ids = set(students["student_id"])

orphan_mask = ~results_clean["student_id"].isin(valid_student_ids)

print("Orphan student rows:", orphan_mask.sum())

results_clean.loc[
    orphan_mask,
    ["result_id", "student_id", "subject_id", "teacher_id", "term"]
].head(10)

Orphan student rows: 75


,result_id,student_id,subject_id,teacher_id,term
140,9324,90027,2,38,Term 3
613,12630,90016,9,44,Term 1
622,17627,90063,10,9,Term 2
1045,9426,90050,4,45,Term 1
1099,10385,90037,9,13,Term 1
1763,3365,90031,4,30,Term 1
1855,11930,90013,6,42,Term 1
2461,4998,90024,1,16,Term 2
2547,26005,90067,3,26,Term 1
3388,2942,90005,3,31,Term 2


In [33]:

orphan_results = results_clean.loc[orphan_mask].copy()


results_clean = results_clean.loc[~orphan_mask].copy()

print("Orphan rows saved separately:", len(orphan_results))
print("Clean result rows remaining:", len(results_clean))

Orphan rows saved separately: 0
Clean result rows remaining: 29925


In [17]:
students_clean = students.copy()

In [18]:
students_clean["sex"] = (
    students_clean["sex"]
    .str.strip()
    .str.lower()
    .str.title()
)

students_clean["sex"].value_counts()

,count
sex,
Female,615
Male,585


In [19]:
students_clean["region"] = (
    students_clean["region"]
    .str.strip()
    .str.lower()
    .str.title()
)

students_clean["region"].value_counts(dropna=False)

,count
region,
Greater Accra,147
Eastern,144
Central,143
Ashanti,142
Northern,140
Volta,137
Bono,132
Western,131
None,84


In [20]:
students_clean["enrolled_date_clean"] = students_clean["enrolled_date"].apply(parse_mixed_date)

print(
    students_clean[
        ["enrolled_date", "enrolled_date_clean"]
    ].head(10)
)

print(
    "Unparsed enrolled dates:",
    students_clean["enrolled_date_clean"].isna().sum()
)

  enrolled_date enrolled_date_clean
0   12-Jan-2024          2024-01-12
1    2024-09-15          2024-09-15
2    28/03/2025          2025-03-28
3    2025-03-03          2025-03-03
4    2025-06-14          2025-06-14
5    2024-08-26          2024-08-26
6    28/07/2024          2024-07-28
7    2025-02-16          2025-02-16
8    2023-05-08          2023-05-08
9    2025-01-14          2025-01-14
Unparsed enrolled dates: 0


In [21]:
print("Students rows:", len(students_clean))
print("Missing regions:", students_clean["region"].isna().sum())
print("Sex categories:", students_clean["sex"].unique())
print("Year groups:", students_clean["year_group"].unique())
print("Unparsed enrolled dates:", students_clean["enrolled_date_clean"].isna().sum())

Students rows: 1200
Missing regions: 84
Sex categories: ['Female' 'Male']
Year groups: ['JHS 2' 'JHS 1' 'JHS 3']
Unparsed enrolled dates: 0


In [22]:
print("SUBJECTS")
display(subjects)

print("\nMissing values in subjects:")
print(subjects.isna().sum())

print("\nDuplicate rows in subjects:")
print(subjects.duplicated().sum())


print("\nTEACHERS")
display(teachers)

print("\nMissing values in teachers:")
print(teachers.isna().sum())

print("\nDuplicate rows in teachers:")
print(teachers.duplicated().sum())

SUBJECTS


,subject_id,subject_name,is_core
0,1,Mathematics,True
1,2,English,True
2,3,Integrated Science,True
3,4,Social Studies,True
4,5,ICT,False
5,6,French,False
6,7,Creative Arts,False
7,8,Ghanaian Language,False
8,9,Career Technology,False
9,10,Religious & Moral Ed,False



Missing values in subjects:
subject_id      0
subject_name    0
is_core         0
dtype: int64

Duplicate rows in subjects:
0

TEACHERS


,teacher_id,teacher_name,subject_id,years_teaching
0,1,Kwesi Oppong,1,8
1,2,Prince Adjei,3,17
2,3,Isaac Quaye,9,30
3,4,Mary Baidoo,1,8
4,5,Esi Gyasi,10,15
5,6,Esi Yeboah,3,8
6,7,Michael Aidoo,3,25
7,8,Sandra Baidoo,7,2
8,9,Ama Oppong,9,30
9,10,Mary Quaye,8,11



Missing values in teachers:
teacher_id        0
teacher_name      0
subject_id        0
years_teaching    0
dtype: int64

Duplicate rows in teachers:
0


In [23]:
teachers_clean = teachers.copy()


In [24]:
print("Missing values in subjects:")
print(subjects.isna().sum())

print("\nDuplicate rows in subjects:")
print(subjects.duplicated().sum())

display(subjects)

Missing values in subjects:
subject_id      0
subject_name    0
is_core         0
dtype: int64

Duplicate rows in subjects:
0


,subject_id,subject_name,is_core
0,1,Mathematics,True
1,2,English,True
2,3,Integrated Science,True
3,4,Social Studies,True
4,5,ICT,False
5,6,French,False
6,7,Creative Arts,False
7,8,Ghanaian Language,False
8,9,Career Technology,False
9,10,Religious & Moral Ed,False


In [25]:
subjects_clean = subjects.copy()

In [26]:
subjects_clean = subjects.copy()
teachers_clean = teachers.copy()

In [27]:
print("===== RESULTS VALIDATION =====")
print("Rows:", len(results_clean))
print("Missing scores:", results_clean["score"].isna().sum())
print("Invalid scores:", ((results_clean["score"] < 0) | (results_clean["score"] > 100)).sum())
print("Unparsed exam dates:", results_clean["exam_date_clean"].isna().sum())
print("Term categories:", sorted(results_clean["term"].dropna().unique()))
print("Duplicate result rows:", results_clean.duplicated(
    subset=[
        "exam_date_clean",
        "student_id",
        "subject_id",
        "teacher_id",
        "term",
        "score",
        "attendance_pct"
    ]
).sum())

print("\n===== STUDENTS VALIDATION =====")
print("Rows:", len(students_clean))
print("Missing regions:", students_clean["region"].isna().sum())
print("Sex categories:", sorted(students_clean["sex"].dropna().unique()))
print("Year groups:", sorted(students_clean["year_group"].dropna().unique()))
print("Unparsed enrolled dates:", students_clean["enrolled_date_clean"].isna().sum())
print("Duplicate student rows:", students_clean.duplicated(
    subset=[
        "student_name",
        "sex",
        "year_group",
        "region",
        "enrolled_date_clean"
    ]
).sum())

print("\n===== SUBJECTS VALIDATION =====")
print("Rows:", len(subjects_clean))
print("Missing values:", subjects_clean.isna().sum().sum())
print("Duplicate rows:", subjects_clean.duplicated().sum())

print("\n===== TEACHERS VALIDATION =====")
print("Rows:", len(teachers_clean))
print("Missing values:", teachers_clean.isna().sum().sum())
print("Duplicate rows:", teachers_clean.duplicated().sum())

print("\n===== RELATIONSHIP VALIDATION =====")
print(
    "Orphan students:",
    (~results_clean["student_id"].isin(students_clean["student_id"])).sum()
)
print(
    "Orphan subjects:",
    (~results_clean["subject_id"].isin(subjects_clean["subject_id"])).sum()
)
print(
    "Orphan teachers:",
    (~results_clean["teacher_id"].isin(teachers_clean["teacher_id"])).sum()
)

===== RESULTS VALIDATION =====
Rows: 29925
Missing scores: 1551
Invalid scores: 0
Unparsed exam dates: 0
Term categories: ['Term 1', 'Term 2', 'Term 3']
Duplicate result rows: 0

===== STUDENTS VALIDATION =====
Rows: 1200
Missing regions: 84
Sex categories: ['Female', 'Male']
Year groups: ['JHS 1', 'JHS 2', 'JHS 3']
Unparsed enrolled dates: 0
Duplicate student rows: 0

===== SUBJECTS VALIDATION =====
Rows: 10
Missing values: 0
Duplicate rows: 0

===== TEACHERS VALIDATION =====
Rows: 50
Missing values: 0
Duplicate rows: 0

===== RELATIONSHIP VALIDATION =====
Orphan students: 0
Orphan subjects: 0
Orphan teachers: 0


In [28]:
results_final = (
    results_clean
    .drop(columns=["exam_date"])
    .rename(columns={"exam_date_clean": "exam_date"})
    .copy()
)

students_final = (
    students_clean
    .drop(columns=["enrolled_date"])
    .rename(columns={"enrolled_date_clean": "enrolled_date"})
    .copy()
)

subjects_final = subjects_clean.copy()
teachers_final = teachers_clean.copy()

print("Results:", results_final.shape)
print("Students:", students_final.shape)
print("Subjects:", subjects_final.shape)
print("Teachers:", teachers_final.shape)

display(results_final.head())
display(students_final.head())

Results: (29925, 8)
Students: (1200, 6)
Subjects: (10, 3)
Teachers: (50, 4)


,result_id,student_id,subject_id,teacher_id,term,score,attendance_pct,exam_date
0,5286,633,9,44,Term 1,NaN,75.1,2024-01-05
1,11177,421,9,50,Term 2,57.1,94.4,2025-05-18
2,10157,222,3,22,Term 3,48.9,83.2,2024-05-04
3,16448,961,10,42,Term 3,41.5,94.4,2025-06-21
4,29013,190,9,23,Term 3,27.2,94.4,2025-09-29


,student_id,student_name,sex,year_group,region,enrolled_date
0,1,Nana Agyemang,Female,JHS 2,Eastern,2024-01-12
1,2,Fatima Nkrumah,Male,JHS 2,Eastern,2024-09-15
2,3,Grace Bediako,Male,JHS 1,Greater Accra,2025-03-28
3,4,Afua Appiah,Male,JHS 3,Central,2025-03-03
4,5,Eric Bediako,Male,JHS 2,Ashanti,2025-06-14


In [29]:
!pip install sqlalchemy

In [30]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

safe_password = quote_plus(password)

engine = create_engine(
    f"postgresql+psycopg2://{username}:{safe_password}@{host}:{port}/{database}",
    connect_args={"sslmode": "require"}
)

print("SQLAlchemy engine ready.")

SQLAlchemy engine ready.


In [31]:
results_final.to_sql(
    "results_clean",
    engine,
    schema="group3a",
    if_exists="replace",
    index=False
)

students_final.to_sql(
    "students_clean",
    engine,
    schema="group3a",
    if_exists="replace",
    index=False
)

subjects_final.to_sql(
    "subjects_clean",
    engine,
    schema="group3a",
    if_exists="replace",
    index=False
)

teachers_final.to_sql(
    "teachers_clean",
    engine,
    schema="group3a",
    if_exists="replace",
    index=False
)

print("All cleaned tables written successfully to group3a.")

All cleaned tables written successfully to group3a.


In [32]:
check_query = """
SELECT
    (SELECT COUNT(*) FROM group3a.results_clean) AS results_rows,
    (SELECT COUNT(*) FROM group3a.students_clean) AS students_rows,
    (SELECT COUNT(*) FROM group3a.subjects_clean) AS subjects_rows,
    (SELECT COUNT(*) FROM group3a.teachers_clean) AS teachers_rows;
"""

pd.read_sql(check_query, engine)

,results_rows,students_rows,subjects_rows,teachers_rows
0,29925,1200,10,50


# Cleaning Summary

The Week 3 cleaning process successfully prepared the school dataset for modelling.

## Results Table
- Original rows: 30,120
- Duplicate rows removed: 120
- Orphan student rows removed from the modelling dataset: 75
- Final cleaned rows: 29,925
- Invalid scores remaining: 0
- Unparsed exam dates: 0
- Duplicate result rows remaining: 0
- Orphan student, subject and teacher keys remaining: 0

## Students Table
- Final rows: 1,200
- Sex standardized to Female and Male
- Region names standardized
- Missing region values retained: 84
- Enrolled dates successfully converted
- Unparsed enrolled dates: 0
- Duplicate student rows: 0

## Subjects Table
- Final rows: 10
- Missing values: 0
- Duplicate rows: 0

## Teachers Table
- Final rows: 50
- Missing values: 0
- Duplicate rows: 0

## Database Output
The cleaned tables were written successfully to:

- `group3a.results_clean`
- `group3a.students_clean`
- `group3a.subjects_clean`
- `group3a.teachers_clean`

These tables are now ready for Week 4 data modelling in Power BI.